# Size-definition comparison

Three options, with all other six inputs, response definitions, forest settings and weighting unchanged:

1. **Current:** $S_L=\mu_{L,p}^{D}/1\,\mathrm{km}$.
2. **No size:** remove $S_L$.
3. **Edited footprint:** $S_{\mathrm{edit}}=\sum_{e\in E_{\mathrm{edit}}}r_e^{D}L_e/1\,\mathrm{km}$.

The third option retains the existing containing-domain width-based visit probabilities $r_e^D$; it does not renormalize probabilities at the selected unit entrance. Selected reaches come from the saved parent unit membership, not a coordinate-difference heuristic. Therefore the footprint need not itself be closed: the surrounding connected domain still supplies probabilities. Unchanged inter-block connectors and tails remain excluded for the new pairs and segmented cases, whose edited sizes are already calculated.

All 2,170 original predictor records join uniquely to saved membership and domain graphs. There are no unmatched records. Sizes change for 1,719 records; the 258 simple-unit sizes are unchanged. The two previously excluded original predictor records remain excluded. All other predictors and responses are verified identical to the previous experiment.

Two training cohorts are tested: original only, and original plus six pairs plus 15 segmented centerlines. Each holds out all comparisons from an entire network. This is 60 new fixed-setting fits, reusing the 30 matching current-size fits from the preceding experiment. It is exploratory follow-up after inspecting network 22, not independent nested model selection. No final model is promoted.

Training target: full excess-volume centroid shift. Peak timing is a transfer diagnostic. Scores use equal network weight within each evaluation cohort; MAE/RMSE are minutes, relative error uses a one-minute denominator floor. Small pair-cohort results are exploratory (six cases in three networks).


## Result and recommendation

Keep the frozen model. Current size remains the best of these choices when trained on the original data: R² 0.615, versus 0.596 for edited size and 0.263 without size. Corresponding floored relative errors are 63.74%, 64.99% and 103.63%.

With both additions in training, edited size removes most of the network-22 catastrophe: its original-case MAE falls from 202.40 to 4.68 minutes. Across all original cases, however, R² is only 0.463 and relative error 68.07%, compared with the unchanged baseline's 0.615 and 63.74%. No size gives R² −0.074 and relative error 102.34%. The targeted no-size success in network 22 does not generalize to all networks.

Keep evaluation cohorts separate. With edited size and augmented training, six-pair R² is −5.339 and segmented-case R² is 0.042. The largest pair in network 22 is overpredicted: actual centroid shift −1.991 hours, prediction −5.946 hours. The original-only/current-size model predicted the pair cohort with R² 0.881. Thus correcting the size definition addresses the identified failure mechanism but does not establish that whole-network training augmentation is beneficial.

These tests include **both** six pairs and 15 centerlines in the augmented cohort; the previous centerline-only figures therefore differ slightly. Further work should retain segmented centerlines as a stress-test cohort and expand the adjacent-unit pilot separately. Do not infer a statistically significant ranking from these point estimates or change the final model on their basis alone.


In [3]:
from pathlib import Path
import pandas as pd
import plotly.express as px
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').is_dir())
OUT = ROOT/'RAPID/outputs/hydrograph_closed_unit_extension_20261019/size_comparison'
scores = pd.read_csv(OUT/'scores.csv')
main = scores.query("target == 'centroid_shift_hours' and subset == 'all'")
display(main[['training','size','cohort','n','r2','mae','relative_error_pct']].round(3))

,training,size,cohort,n,r2,mae,relative_error_pct
0,augmented,current,original,2170,-25.486,16.313,1235.639
6,augmented,current,pair,6,-12.084,65.290,93.760
12,augmented,current,segmented,15,0.186,226.542,51.317
18,augmented,edited,original,2170,0.463,3.045,68.075
24,augmented,edited,pair,6,-5.339,47.692,78.509
30,augmented,edited,segmented,15,0.042,227.351,47.082
36,augmented,none,original,2170,-0.074,4.662,102.345
42,augmented,none,pair,6,-1.871,55.306,252.905
48,augmented,none,segmented,15,-0.640,340.359,88.651
54,original_only,current,original,2170,0.615,2.823,63.741


In [4]:
network = pd.read_csv(OUT/'network_scores.csv', dtype={'example':str})
z = network.query("target == 'centroid_shift_hours' and cohort == 'original'")
display(z.pivot(index='example',columns=['training','size'],values='mae_minutes').round(3))
px.bar(z, x='example', y='mae_minutes', color='size', facet_row='training', barmode='group',
       title='Original-case MAE by held-out network', labels={'mae_minutes':'MAE (minutes)'}).show()

training augmented               original_only              
size       current edited   none       current edited   none
example                                                     
02           1.624  1.704  3.546         1.271  1.702  2.747
03           1.596  1.923  2.521         1.430  1.677  2.925
05           5.163  5.346  5.960         4.935  4.960  6.317
13           2.394  2.222  4.265         2.399  2.096  3.954
15           1.935  1.981  2.310         1.914  1.904  2.148
21           2.998  3.282  3.437         3.347  3.648  4.297
22         202.395  4.680  2.039         1.726  2.224  1.650
37           2.804  2.621  3.140         2.866  2.978  3.013
41           3.475  3.544  4.524         3.484  3.494  4.480
45           3.361  3.119  4.847         2.840  2.932  4.477
46           4.624  4.710  6.523         4.673  4.806  6.192
48           2.747  2.064  7.932         2.335  2.111  4.759
54           2.491  1.976  4.981         2.572  2.310  5.363
55           4.030  4.120  4.847         4.015  4.106  4.708
56           3.054  2.377  9.062         2.541  2.406  8.814

In [5]:
audit = pd.read_csv(OUT/'footprint_audit.csv', dtype={'example':str})
display(audit.groupby('example')[['domain_size_current','domain_size_edited']].median().round(3))
px.scatter(audit, x='domain_size_current', y='domain_size_edited', color='example',
           log_x=True, log_y=True, hover_data=['state','edited_fraction','scope'],
           title='Current size versus edited-footprint size (units of 1 km)').show()

,domain_size_current,domain_size_edited
example,,
02,4.252,0.861
03,2.930,0.943
05,2.242,1.689
13,5.133,1.863
15,42.560,0.430
21,3.150,2.139
22,91.683,0.668
37,89.967,0.428
41,12.505,2.212


The full tables also include evaluation-only exclusions of network 22 and example 54, and peak-timing transfer scores. Exclusion tables do not represent retraining with those networks removed. Source hashes and settings are in `design.json`; membership/graph hashes are in `footprint_sources.json`.

Reproduce/resume: `python RAPID/investigations/closed_unit_extension_20261019/size_comparison.py` using the project environment. Saved checkpoints are fingerprinted. Frozen-model artifacts are untouched.